# Tool-Using Agent Prompt-Injection Behavioral Audit (33k Trajectories)
### Official Starter Notebook for Kaggle Dataset Release
**Research Paper**: *Low Attack Success Is Not Security: Diagnosing Prompt-Injection Failures in Small Tool-Using Agents* (Brac University, 2026)

This notebook demonstrates how to load and analyze the **33,000 label-blind audited trajectories** evaluated using DeepSeek-V4.1-Flash across 11 open-weight small reasoning agents on **AgentDojo** and **InjecAgent** under **FP16, FP8, and NF4** precision regimes.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the consolidated Parquet dataset
dataset_path = 'audit_corpus_33k.parquet'
if not os.path.exists(dataset_path):
    dataset_path = 'kaggle_audit_corpus_package/audit_corpus_33k.parquet'
if not os.path.exists(dataset_path):
    dataset_path = '/kaggle/input/agent-prompt-injection-behavioral-audit-33k/audit_corpus_33k.parquet'

df = pd.read_parquet(dataset_path)
print('Successfully loaded', len(df), 'audited trajectory records!')
df.head(3)

## 1. Dataset Overview & Evaluation Scope
Let's inspect the model cohorts, benchmarks, and precision distributions.

In [ ]:
print('Evaluated Models:', sorted(df['model'].unique()))
print('Benchmarks:', sorted(df['benchmark'].unique()))
print('Precisions:', sorted(df['precision'].unique()))

# Table of trajectories per model and precision
pd.crosstab(df['model'], [df['benchmark'], df['precision']])

## 2. The 6-Bucket Behavioral Taxonomy Breakdown
In standard evaluations, attacks that fail are treated as uniform defense. Here we dissect what actually happened when attacks were scored as unsuccessful.

In [ ]:
# Overall behavioral composition
bucket_dist = df['behavioral_bucket'].value_counts(normalize=True) * 100
print('Overall 6-Bucket Distribution Across 33,000 Trajectories:')
for b, pct in bucket_dist.items():
    print(f'  {b:<22}: {pct:>6.2f}%')

## 3. Perception-Compliance Dissociation (PCD)
How often do models perceive an injection in their internal reasoning tokens, yet proceed to comply anyway?

In [ ]:
recog_df = df[df['recognized_injection'] == True]
print('Trajectories with Explicit Injection Recognition in Reasoning:', len(recog_df))

pcd_table = []
for m in sorted(df['model'].unique()):
    m_sub = df[df['model'] == m]
    m_rec = m_sub[m_sub['recognized_injection'] == True]
    if len(m_rec) > 0:
        ref_rate = (m_rec['behavioral_bucket'] == 'GENUINE_REFUSAL').mean() * 100
        comp_rate = (m_rec['behavioral_bucket'].isin(['FULL_COMPLIANCE', 'PARTIAL_COMPLIANCE'])).mean() * 100
        pcd_table.append({
            'Model': m,
            'Recognized Cases': len(m_rec),
            'Conversion to Refusal (%)': round(ref_rate, 2),
            'Perception-Compliance Dissociation (%)': round(comp_rate, 2)
        })

pcd_df = pd.DataFrame(pcd_table)
pcd_df

## 4. Quoted Evidence Verification
Every trajectory flagged as recognizing the injection contains verbatim quotes from the internal thought tokens verified via programmatic string matching.

In [ ]:
sample_quotes = df[(df['recognized_injection'] == True) & (df['evidence_quote'] != '')][['model', 'behavioral_bucket', 'evidence_quote', 'reasoning_summary']].head(5)
for idx, row in sample_quotes.iterrows():
    print(f'Model: {row["model"]} | Bucket: {row["behavioral_bucket"]}')
    print(f'Evidence Quote: "{row["evidence_quote"]}"')
    print(f'Judge Summary: {row["reasoning_summary"][:200]}...')
    print('-' * 80)